# GPSseq · Allen atlas preparation

This notebook downloads public **Allen mouse brain atlas** files and optionally renames 25-µm structure masks using `brain_region_annotations.csv`.

**Before running**

1. From the GPSseq repository root, create/activate your Python 3.9 environment and install `requirements.txt`, then complete `tutorials/01_install_environment.ipynb`.
2. Open this notebook from the repository (`tutorials/02_prepare_allen_atlas.ipynb`).
3. Place **your own** `brain_region_annotations.csv` in `brain_registration/metadata/Allen_nrrd/` (this file is not provided by the Allen URLs below).
4. Run the cells in order. First inspect the list of mask files and available disk space. In the configuration cell, change `RUN_DOWNLOADS = False` to `True` to start the actual downloads.

**Important:** The `structure_masks_25/` folder contains *hundreds of files*, and the 10-µm average template is itself large. Make sure you have sufficient disk space and a stable Internet connection. Re-running normally skips valid already-downloaded files. Downloads can take a long time. This notebook **does not download trained GPSseq model weights**; that requires a separate link from the model authors.

**Sources:** [Allen Brain Cell Atlas NIfTI objects](https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com/index.html#image_volumes/Allen-CCF-2020/20250331/) · [Allen 25-µm NRRD mask directory](https://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/structure_masks/structure_masks_25/)

## 1. Configure output folders

The notebook finds the GPSseq project root by searching the current directory and its parents. The directory must contain `brain_registration/` and `requirements.txt`.

Files will be saved in:

- `brain_registration/metadata/Allen_nii/`: three 10-µm `.nii.gz` files.
- `brain_registration/metadata/Allen_nrrd/`: two 25-µm `.nrrd` atlas files, plus your CSV.
- `brain_registration/metadata/Allen_nrrd/mask/`: all 25-µm `structure_<ID>.nrrd` masks, renamed to `<ARA abbreviation>.nrrd` where a unique, safe match is found.

For an initial small-scale test, set `MASK_LIMIT = 5`. Set it back to `None` for **all** masks. `RUN_DOWNLOADS` starts as `False`. Set it to `True` only after checking your download targets and disk space.

In [ ]:
from pathlib import Path
import csv
import os
import re
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from html.parser import HTMLParser
from urllib.parse import urljoin, urlparse
from urllib.request import Request, urlopen

def extract_structure_id(filename):
    """Return the numeric ID in structure_<ID>.nrrd, or None for other names.

    Example: structure_1014.nrrd -> '1014'.
    """
    match = re.fullmatch(r'structure_(\d+)\.nrrd', Path(filename).name)
    return str(int(match.group(1))) if match else None


RUN_DOWNLOADS = False      # SAFE DEFAULT: change to True only when ready to download.
DOWNLOAD_NIFTI = True
DOWNLOAD_NRRD = True
DOWNLOAD_MASKS = True
RENAME_MASKS = True
MASK_LIMIT = None           # e.g. 5 for a quick test; None downloads every mask.
MAX_WORKERS = 3             # Be considerate of the public server.
MAX_RETRIES = 3
CHUNK_SIZE = 1024 * 1024


def find_project_root(start):
    start = Path(start).resolve()
    for parent in (start, *start.parents):
        if (parent / 'brain_registration').is_dir() and (parent / 'requirements.txt').is_file():
            return parent
    raise FileNotFoundError(
        'Cannot locate the  GPSseq repository. Start Jupyter from GPSseq/ '
        '(with brain_registration/ and requirements.txt present).'
    )

ROOT = find_project_root(Path.cwd())
METADATA_DIR = ROOT / 'brain_registration' / 'metadata'
NIFTI_DIR = METADATA_DIR / 'Allen_nii'
NRRD_DIR = METADATA_DIR / 'Allen_nrrd'
MASK_DIR = NRRD_DIR / 'mask'
CSV_PATH = NRRD_DIR / 'brain_region_annotations.csv'
MANIFEST_PATH = NRRD_DIR / 'mask_rename_manifest.csv'

for folder in (NIFTI_DIR, NRRD_DIR, MASK_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('GPSseq root:', ROOT)
print('CSV file:', CSV_PATH, '| exists:', CSV_PATH.is_file())
print('NIfTI output:', NIFTI_DIR)
print('NRRD output:', NRRD_DIR)
print('Mask output:', MASK_DIR)
print('Actual downloads enabled:', RUN_DOWNLOADS)

## 2. Declare the official URLs

The three NIfTI files use the direct S3 **object URLs**, not the website explorer address containing `#` (browser fragments are not part of an HTTP download URL).

In [ ]:
S3_ROOT = (
    'https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com/'
    'image_volumes/Allen-CCF-2020/20250331/'
)
CCF_ROOT = (
    'https://download.alleninstitute.org/informatics-archive/current-release/'
    'mouse_ccf/'
)
MASK_INDEX_URL = (
    CCF_ROOT + 'annotation/ccf_2017/structure_masks/structure_masks_25/'
)

NIFTI_FILES = {
    'annotation_10.nii.gz': S3_ROOT + 'annotation_10.nii.gz',
    'average_template_10.nii.gz': S3_ROOT + 'average_template_10.nii.gz',
    'annotation_boundary_10.nii.gz': S3_ROOT + 'annotation_boundary_10.nii.gz',
}
NRRD_FILES = {
    'average_template_25.nrrd': CCF_ROOT + 'average_template/average_template_25.nrrd',
    'annotation_25.nrrd': CCF_ROOT + 'annotation/ccf_2017/annotation_25.nrrd',
}
print('NIfTI URLs:', len(NIFTI_FILES))
print('NRRD URLs:', len(NRRD_FILES))

## 3. Collect every mask filename from the Allen directory listing

The source contains files with names such as `structure_1000.nrrd`. The mask's **numeric ID** is what matches the CSV: `structure_1014.nrrd` → `1014`. This code parses the Allen HTML index and accepts **only direct `structure_<integer>.nrrd` links** from that directory. The resulting list is sorted by structure ID.

In [ ]:
class AnchorCollector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == 'a':
            href = dict(attrs).get('href')
            if href:
                self.links.append(href)


def open_http(url, timeout=60):
    return urlopen(Request(url, headers={'User-Agent': 'GPSseq-atlas-setup/1.0'}), timeout=timeout)


def discover_mask_files(index_url):
    with open_http(index_url, timeout=90) as response:
        html = response.read().decode('utf-8', errors='replace')
    parser = AnchorCollector()
    parser.feed(html)
    base = urlparse(index_url)
    base_dir = base.path.rstrip('/') + '/'
    found = {}
    for href in parser.links:
        url = urljoin(index_url, href)
        parsed = urlparse(url)
        filename = Path(parsed.path).name
        structure_id = extract_structure_id(filename)
        if structure_id is None:
            continue
        # Accept only files inside the expected folder, not arbitrary linked domains/paths.
        if parsed.netloc != base.netloc or parsed.path != base_dir + filename:
            continue
        found[filename] = url
    if not found:
        raise RuntimeError(
            'No structure_*.nrrd links found. The Allen directory listing may have changed; '
            'open MASK_INDEX_URL in a browser and check its content.'
        )
    return dict(sorted(found.items(), key=lambda item: int(extract_structure_id(item[0]))))

mask_files = discover_mask_files(MASK_INDEX_URL) if DOWNLOAD_MASKS else {}
print('Found masks in the official index:', len(mask_files))
print('First 12:', list(mask_files)[:12])
if MASK_LIMIT is not None:
    if MASK_LIMIT < 1:
        raise ValueError('MASK_LIMIT must be positive or None.')
    mask_files = dict(list(mask_files.items())[:MASK_LIMIT])
    print('TEST MODE — selected mask count:', len(mask_files))

print('Planned individual files:',
      (len(NIFTI_FILES) if DOWNLOAD_NIFTI else 0)
      + (len(NRRD_FILES) if DOWNLOAD_NRRD else 0)
      + len(mask_files))
print('RUN_DOWNLOADS =', RUN_DOWNLOADS)

## 4. Download and verify atlas files

Downloads are streamed to temporary `*.part` files and only renamed to the final target once complete. Invalid/non-data responses are rejected using the NRRD or gzip file header; existing valid files are skipped. Failed downloads are retried. Mask renaming is performed separately after downloading.

**This cell is the long-running step.** To execute, set `RUN_DOWNLOADS = True` in Section 1 and run the notebook again. For a quick smoke test, first use `MASK_LIMIT = 5` and subsequently reset it to `None`.

In [ ]:
def header_is_valid(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return False
    with path.open('rb') as handle:
        prefix = handle.read(8)
    if path.name.endswith('.nii.gz'):
        return prefix.startswith(b'\x1f\x8b')
    if path.suffix.lower() == '.nrrd':
        return prefix.startswith(b'NRRD')
    return False


def download_file(url, destination, retries=MAX_RETRIES):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if header_is_valid(destination):
        return 'already_present', destination.name

    partial = destination.with_name(destination.name + '.part')
    for attempt in range(1, retries + 1):
        try:
            if partial.exists():
                partial.unlink()  # Retry from scratch; incomplete downloads are never accepted.
            with open_http(url, timeout=120) as response:
                expected = response.headers.get('Content-Length')
                downloaded = 0
                with partial.open('wb') as output:
                    first_chunk = response.read(CHUNK_SIZE)
                    if not first_chunk:
                        raise IOError('Empty HTTP response')
                    if destination.name.endswith('.nii.gz') and not first_chunk.startswith(b'\x1f\x8b'):
                        raise IOError('Unexpected content (not gzip NIfTI)')
                    if destination.suffix.lower() == '.nrrd' and not first_chunk.startswith(b'NRRD'):
                        raise IOError('Unexpected content (not NRRD)')
                    output.write(first_chunk)
                    downloaded += len(first_chunk)
                    while True:
                        chunk = response.read(CHUNK_SIZE)
                        if not chunk:
                            break
                        output.write(chunk)
                        downloaded += len(chunk)
                if expected is not None and downloaded != int(expected):
                    raise IOError(f'Incomplete transfer: {downloaded} vs {expected} bytes')
            os.replace(str(partial), str(destination))
            return 'downloaded', destination.name
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if attempt == retries:
                return 'failed', f'{destination.name}: {exc}'
            time.sleep(min(2 ** attempt, 10))


def load_completed_rename_map(manifest_path):
    """IDs renamed in a prior run (for idempotent re-runs)."""
    completed = {}
    if not manifest_path.is_file():
        return completed
    with manifest_path.open('r', encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            if row.get('status') in ('renamed', 'already_renamed'):
                mask_id = extract_structure_id(row.get('original_name', ''))
                if mask_id is not None and row.get('renamed_name'):
                    completed[mask_id] = row['renamed_name']
    return completed


def download_masks(mask_urls, destination_dir, workers=MAX_WORKERS):
    previously_renamed = load_completed_rename_map(MANIFEST_PATH)
    jobs = []
    already_renamed = 0
    for filename, url in mask_urls.items():
        mask_id = extract_structure_id(filename)
        if mask_id is None:
            continue
        existing = previously_renamed.get(mask_id)
        if existing and header_is_valid(destination_dir / existing):
            already_renamed += 1
        else:
            jobs.append((url, destination_dir / filename))
    print(f'Already renamed in earlier runs: {already_renamed}; to check/download: {len(jobs)}')
    counts = Counter()
    failures = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(download_file, url, path) for url, path in jobs]
        for i, future in enumerate(as_completed(futures), 1):
            status, message = future.result()
            counts[status] += 1
            if status == 'failed':
                failures.append(message)
                print('FAILED:', message)
            if i % 20 == 0 or i == len(futures):
                print(f'Mask progress: {i}/{len(futures)} | {dict(counts)}')
    return counts, failures


if not RUN_DOWNLOADS:
    print('Download preview only. Change RUN_DOWNLOADS=True in Section 1 to download.')
else:
    for enabled, url_map, destination_dir, description in (
        (DOWNLOAD_NIFTI, NIFTI_FILES, NIFTI_DIR, '10-µm NIfTI'),
        (DOWNLOAD_NRRD, NRRD_FILES, NRRD_DIR, '25-µm NRRD'),
    ):
        if enabled:
            print('\nDownloading:', description)
            for filename, url in url_map.items():
                status, detail = download_file(url, destination_dir / filename)
                print(status, detail)
                if status == 'failed':
                    raise RuntimeError(f'Failed to download {filename}. Re-run this cell to retry.')
    if DOWNLOAD_MASKS:
        print('\nDownloading 25-µm region masks ...')
        counts, failures = download_masks(mask_files, MASK_DIR)
        print('Mask summary:', dict(counts))
        if failures:
            print('Some mask downloads failed. Re-run this cell to retry them.')

In [ ]:
def normalize_id(raw):
    raw = str(raw).strip()
    if re.fullmatch(r'\d+(?:\.0+)?', raw):
        return str(int(float(raw)))
    return None


def valid_filename_stem(value):
    value = str(value).strip()
    if not value or value in ('.', '..'):
        return False
    if re.search(r'[<>:"/\\|?*\x00-\x1f]', value):
        return False
    if value[-1] in (' ', '.'):
        return False
    if value.upper().split('.')[0] in {
        'CON', 'PRN', 'AUX', 'NUL', *(f'COM{i}' for i in range(1, 10)),
        *(f'LPT{i}' for i in range(1, 10))
    }:
        return False
    return True


def read_region_mapping(csv_path):
    if not csv_path.is_file():
        raise FileNotFoundError(
            f'Missing {csv_path}. Put brain_region_annotations.csv there first. '
            'It is NOT among the Allen files downloaded above.'
        )
    mapping = {}
    ambiguous_ids = set()
    invalid_rows = []
    with csv_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError('Annotation CSV is empty.')
        col_lookup = {' '.join(str(name).split()).casefold(): name for name in reader.fieldnames}
        id_col = col_lookup.get('ara structure id')
        abbr_col = col_lookup.get('ara abbreviation')
        if id_col is None or abbr_col is None:
            raise ValueError(
                'Required CSV columns: "ARA structure ID" and "ARA abbreviation"; '
                f'found {reader.fieldnames}'
            )
        for line_no, row in enumerate(reader, start=2):
            region_id = normalize_id(row.get(id_col, ''))
            abbr = str(row.get(abbr_col, '') or '').strip()
            if region_id is None or not valid_filename_stem(abbr):
                invalid_rows.append(line_no)
                continue
            if region_id in mapping and mapping[region_id] != abbr:
                ambiguous_ids.add(region_id)
            else:
                mapping[region_id] = abbr
    for region_id in ambiguous_ids:
        mapping.pop(region_id, None)
    print('Valid unambiguous ID mappings:', len(mapping))
    print('Conflicting duplicated IDs:', len(ambiguous_ids))
    print('CSV rows with missing/invalid IDs or names:', len(invalid_rows))
    return mapping, ambiguous_ids


def rename_mask_files(mask_dir, csv_path, manifest_path):
    mapping, ambiguous_ids = read_region_mapping(csv_path)
    print('Example: structure_1014.nrrd -> structure ID', extract_structure_id('structure_1014.nrrd'),
          '-> CSV abbreviation:', mapping.get('1014', '(no match in CSV)'))
    filenames = sorted(mask_dir.glob('structure_*.nrrd'))
    candidate_ids = []
    for path in filenames:
        region_id = extract_structure_id(path.name)
        if region_id is not None:
            candidate_ids.append((path, region_id))

    # Consider ALL Allen mask IDs, even when a small test subset has been downloaded.
    # This prevents future collisions if two IDs map to the same abbreviation.
    collisions = defaultdict(list)
    for region_id, abbr in mapping.items():
        collisions[abbr.casefold()].append(region_id)
    ambiguous_names = {name for name, ids in collisions.items() if len(ids) > 1}
    rows = []
    for path, region_id in candidate_ids:
        abbr = mapping.get(region_id)
        target_name = f'{abbr}.nrrd' if abbr else ''
        if region_id in ambiguous_ids:
            status = 'conflicting_id'
        elif not abbr:
            status = 'unmapped'
        elif abbr.casefold() in ambiguous_names:
            status = 'conflicting_abbreviation'
        elif (mask_dir / target_name).exists():
            status = 'target_already_exists'
        else:
            (mask_dir / target_name).parent.mkdir(parents=True, exist_ok=True)
            path.rename(mask_dir / target_name)
            status = 'renamed'
        rows.append({
            'structure_id': region_id, 'original_name': path.name,
            'renamed_name': target_name, 'status': status,
        })

    # Keep previously successful rename records, so a second run does not download
    # the original structure-ID files again.
    previous_rows = []
    if manifest_path.is_file():
        with manifest_path.open('r', encoding='utf-8-sig', newline='') as f:
            previous_rows = list(csv.DictReader(f))
    by_id = {r['structure_id']: r for r in previous_rows if r.get('structure_id')}
    for row in rows:
        by_id[row['structure_id']] = row
    for row in by_id.values():
        if (row['status'] == 'renamed' and
                not (mask_dir / row.get('renamed_name', '')).is_file()):
            row['status'] = 'missing_after_rename'
    with manifest_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['structure_id', 'original_name', 'renamed_name', 'status'])
        writer.writeheader()
        writer.writerows(sorted(by_id.values(), key=lambda r: int(r['structure_id'])))
    status_counts = Counter(r['status'] for r in rows)
    print('Rename outcomes this run:', dict(status_counts))
    print('Manifest:', manifest_path)
    for row in rows[:10]:
        print(row['original_name'], '→', row['renamed_name'] or '(kept original)', '[', row['status'], ']')
    return rows


if not RUN_DOWNLOADS:
    print('Rename preview only: set RUN_DOWNLOADS=True in Section 1 to perform renaming.')
elif RENAME_MASKS:
    rename_rows = rename_mask_files(MASK_DIR, CSV_PATH, MANIFEST_PATH)
else:
    print('RENAME_MASKS=False: keeping original structure-ID filenames.')

## 5. Check downloaded files

This cell only reports local files. The following printout helps verify output locations before running the rest of GPSseq.

In [ ]:
def sample_names(folder, pattern, n=6):
    files = list(folder.glob(pattern))
    return len(files), [f.name for f in files[:n]]

for folder, pattern in (
    (NIFTI_DIR, '*.nii.gz'),
    (NRRD_DIR, '*.nrrd'),
    (MASK_DIR, '*.nrrd'),
):
    count, names = sample_names(folder, pattern)
    print(f'{folder.relative_to(ROOT)}: {count} matching files')
    print('  Examples:', names)
print('CSV:', CSV_PATH.is_file())
print('Manifest:', MANIFEST_PATH.is_file())

## 6. GitHub and reproducibility notes

Do **not** commit downloaded atlas volumes, mask files, or large neural-network weights to GitHub. Keep the downloader notebook, the small CSV mapping file (if redistribution is permitted), and project source code under version control.

Suggested `.gitignore` additions:

```gitignore
brain_registration/metadata/Allen_nii/
brain_registration/metadata/Allen_nrrd/*.nrrd
brain_registration/metadata/Allen_nrrd/mask/
brain_registration/metadata/Allen_nrrd/mask_rename_manifest.csv
```

After you publish model checkpoints using Zenodo, Figshare, Hugging Face, or another approved hosting location, add a **separate** weights-download step to your GPSseq README. Windows EXE users should use the same reference data in the locations expected by the executable (the application resource-path configuration may need adjustment).